In [10]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLEAU_DIR = PROJECT_ROOT / "data" / "tableau"

TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
predictions = pd.read_csv(PROCESSED_DIR / "etf_ff5_with_predictions.csv")
rolling_betas = pd.read_csv(PROCESSED_DIR / "rolling_factor_betas.csv")
regression_summary = pd.read_csv(PROCESSED_DIR / "factor_regression_summary.csv")

predictions["date"] = pd.to_datetime(predictions["date"])
rolling_betas["date"] = pd.to_datetime(rolling_betas["date"])

In [12]:
tableau_df = predictions.merge(
    rolling_betas,
    on=["date", "ticker"],
    how="left"
)

tableau_df.head()

,date,ticker,return,mkt_rf,smb,hml,rmw,cma,rf,excess_return,...,cumulative_residual,rolling_alpha,rolling_beta_market,rolling_beta_smb,rolling_beta_hml,rolling_beta_rmw,rolling_beta_cma,rolling_r_squared,window_months,n_obs
0,2015-02-28,IWM,0.059464,0.0614,0.0036,-0.0179,-0.0110,-0.0175,0.0,0.059464,...,-0.003930,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-03-31,IWM,0.017700,-0.0109,0.0308,-0.0038,0.0007,-0.0062,0.0,0.017700,...,0.000753,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-04-30,IWM,-0.025649,0.0060,-0.0301,0.0180,0.0005,-0.0062,0.0,-0.025649,...,-0.007441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-05-31,IWM,0.022364,0.0138,0.0082,-0.0111,-0.0176,-0.0083,0.0,0.022364,...,-0.005771,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-06-30,IWM,0.007829,-0.0154,0.0290,-0.0082,0.0035,-0.0154,0.0,0.007829,...,-0.004508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
etf_metadata = pd.DataFrame({
    "ticker": ["SPY", "QQQ", "IWM", "VTV", "VUG", "QUAL", "MTUM", "USMV", "VLUE"],
    "fund_name": [
        "SPDR S&P 500 ETF",
        "Invesco QQQ Trust",
        "iShares Russell 2000 ETF",
        "Vanguard Value ETF",
        "Vanguard Growth ETF",
        "iShares MSCI USA Quality Factor ETF",
        "iShares MSCI USA Momentum Factor ETF",
        "iShares MSCI USA Min Vol Factor ETF",
        "iShares MSCI USA Value Factor ETF",
    ],
    "style_category": [
        "Broad Market",
        "Growth / Tech",
        "Small Cap",
        "Value",
        "Growth",
        "Quality",
        "Momentum",
        "Minimum Volatility",
        "Value Factor",
    ],
    "benchmark_proxy": [
        "US Market",
        "Nasdaq 100",
        "Russell 2000",
        "Large-Cap Value",
        "Large-Cap Growth",
        "Quality Factor",
        "Momentum Factor",
        "Minimum Volatility",
        "Value Factor",
    ]
})

tableau_df = tableau_df.merge(
    etf_metadata,
    on="ticker",
    how="left"
)

In [14]:
style_cols = [
    "ticker",
    "beta_market",
    "beta_smb",
    "beta_hml",
    "beta_rmw",
    "beta_cma",
    "r_squared"
]

tableau_df = tableau_df.merge(
    regression_summary[style_cols],
    on="ticker",
    how="left",
    suffixes=("", "_full")
)

In [15]:
def assign_style_label(row):
    if row["beta_smb"] > 0.5:
        return "Small-Cap Tilt"
    elif row["beta_hml"] > 0.2:
        return "Value Tilt"
    elif row["beta_hml"] < -0.2:
        return "Growth Tilt"
    elif row["beta_rmw"] > 0.15:
        return "Quality / Profitability Tilt"
    elif row["beta_market"] < 0.8:
        return "Defensive / Low Beta"
    else:
        return "Broad Market"

tableau_df["factor_style_label"] = tableau_df.apply(assign_style_label, axis=1)

In [16]:
tableau_df = tableau_df.sort_values(["ticker", "date"]).copy()

tableau_df["wealth_index"] = (
    tableau_df
    .groupby("ticker")["return"]
    .transform(lambda x: (1 + x).cumprod())
)

tableau_df["running_peak"] = (
    tableau_df
    .groupby("ticker")["wealth_index"]
    .cummax()
)

tableau_df["drawdown"] = (
    tableau_df["wealth_index"] / tableau_df["running_peak"] - 1
)

In [17]:
final_cols = [
    "date",
    "ticker",
    "fund_name",
    "style_category",
    "benchmark_proxy",
    "factor_style_label",

    "return",
    "excess_return",
    "predicted_excess_return",
    "predicted_return",
    "residual",

    "cumulative_actual_return",
    "cumulative_predicted_return",
    "wealth_index",
    "drawdown",

    "mkt_rf",
    "smb",
    "hml",
    "rmw",
    "cma",
    "rf",

    "beta_market",
    "beta_smb",
    "beta_hml",
    "beta_rmw",
    "beta_cma",
    "r_squared",

    "rolling_alpha",
    "rolling_beta_market",
    "rolling_beta_smb",
    "rolling_beta_hml",
    "rolling_beta_rmw",
    "rolling_beta_cma",
    "rolling_r_squared",
    "window_months",
]

tableau_final = tableau_df[final_cols].copy()

In [18]:
print(tableau_final.shape)
print(tableau_final["ticker"].unique())
print(tableau_final["date"].min())
print(tableau_final["date"].max())

tableau_final.head()

(1197, 35)
['IWM' 'MTUM' 'QQQ' 'QUAL' 'SPY' 'USMV' 'VLUE' 'VTV' 'VUG']
2015-02-28 00:00:00
2026-02-28 00:00:00


,date,ticker,fund_name,style_category,benchmark_proxy,factor_style_label,return,excess_return,predicted_excess_return,predicted_return,...,beta_cma,r_squared,rolling_alpha,rolling_beta_market,rolling_beta_smb,rolling_beta_hml,rolling_beta_rmw,rolling_beta_cma,rolling_r_squared,window_months
0,2015-02-28,IWM,iShares Russell 2000 ETF,Small Cap,Russell 2000,Small-Cap Tilt,0.059464,0.059464,0.063394,0.063394,...,-0.034876,0.990035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-03-31,IWM,iShares Russell 2000 ETF,Small Cap,Russell 2000,Small-Cap Tilt,0.017700,0.017700,0.012998,0.012998,...,-0.034876,0.990035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-04-30,IWM,iShares Russell 2000 ETF,Small Cap,Russell 2000,Small-Cap Tilt,-0.025649,-0.025649,-0.017461,-0.017461,...,-0.034876,0.990035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-05-31,IWM,iShares Russell 2000 ETF,Small Cap,Russell 2000,Small-Cap Tilt,0.022364,0.022364,0.020681,0.020681,...,-0.034876,0.990035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-06-30,IWM,iShares Russell 2000 ETF,Small Cap,Russell 2000,Small-Cap Tilt,0.007829,0.007829,0.006559,0.006559,...,-0.034876,0.990035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
output_path = TABLEAU_DIR / "tableau_factorlens_main.csv"

tableau_final.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /Users/phamthanh/Documents/Project/Financial Market/factorlens_etf_style_drift/data/tableau/tableau_factorlens_main.csv
